# ClinPGx REST API Integration & ETL Pipeline

This notebook demonstrates:
1. Querying pharmacogenomics data from the **ClinPGx API** (`https://api.clinpgx.org/v1`) using the OpenAPI 3.0 specification.
2. Flattening nested JSON responses with Pandas.
3. Handling nested list objects to prevent hash/database serialization errors.
4. Exporting structured clean tables to SQLite databases (`clinpgx_drugs.db` and `clinpgx_genes.db`).

In [ ]:
import requests
import pandas as pd
import sqlite3

BASE_URL = "https://api.clinpgx.org/v1"
print("Ready to query ClinPGx API.")

## 1. Fetch Drug / Chemical Data (`/data/chemical`)

In ClinPGx (PharmGKB schema), pharmaceuticals are queried via `/data/chemical` using the `name` parameter.

In [ ]:
drugs_to_fetch = ["codeine", "fluoxetine", "warfarin", "clopidogrel"]
drug_records = []

for drug in drugs_to_fetch:
    resp = requests.get(f"{BASE_URL}/data/chemical", params={"name": drug})
    if resp.status_code == 200:
        payload = resp.json()
        if "data" in payload and isinstance(payload["data"], list):
            drug_records.extend(payload["data"])
    else:
        print(f"Warning: {drug} returned status {resp.status_code}")

print(f"Successfully fetched {len(drug_records)} chemical records.")

## 2. Normalize and Clean Drugs Data

Convert list fields into clean strings to allow safe deduplication and SQLite storage.

In [ ]:
df_drugs = pd.json_normalize(drug_records)

# Convert list-type columns to comma-separated strings to allow hashing & SQLite storage
for col in df_drugs.columns:
    if df_drugs[col].apply(lambda x: isinstance(x, list)).any():
        df_drugs[col] = df_drugs[col].apply(lambda x: ", ".join(map(str, x)) if isinstance(x, list) else x)

# Deduplicate and fill missing values
df_drugs = df_drugs.drop_duplicates()
df_drugs = df_drugs.fillna("Unknown")

# Normalize column names
df_drugs.columns = [col.strip().lower().replace(" ", "_") for col in df_drugs.columns]

# Save to SQLite
conn = sqlite3.connect("clinpgx_drugs.db")
df_drugs.to_sql("drugs", conn, if_exists="replace", index=False)
conn.close()

print("Saved cleaned drugs dataset to SQLite database: clinpgx_drugs.db")
df_drugs[["id", "name", "types"]].head()

## 3. Fetch Gene Data (`/data/gene`)

Query pharmacogenes (e.g., CYP2D6, CYP2C19, VKORC1) via `/data/gene` using `symbol`.

In [ ]:
genes_to_fetch = ["CYP2D6", "CYP2C19", "VKORC1", "DPYD", "TPMT"]
gene_records = []

for gene in genes_to_fetch:
    resp = requests.get(f"{BASE_URL}/data/gene", params={"symbol": gene})
    if resp.status_code == 200:
        payload = resp.json()
        if "data" in payload and isinstance(payload["data"], list):
            gene_records.extend(payload["data"])

df_genes = pd.json_normalize(gene_records)

# Convert list columns to string
for col in df_genes.columns:
    if df_genes[col].apply(lambda x: isinstance(x, list)).any():
        df_genes[col] = df_genes[col].apply(lambda x: ", ".join(map(str, x)) if isinstance(x, list) else x)

df_genes = df_genes.drop_duplicates().fillna("Unknown")
df_genes.columns = [col.strip().lower().replace(" ", "_") for col in df_genes.columns]

# Save to SQLite
conn = sqlite3.connect("clinpgx_genes.db")
df_genes.to_sql("genes", conn, if_exists="replace", index=False)
conn.close()

print(f"Saved {len(df_genes)} genes to SQLite database: clinpgx_genes.db")
df_genes[["id", "symbol", "name"]].head()

## 4. Verify SQLite Database Tables

In [ ]:
conn = sqlite3.connect("clinpgx_drugs.db")
preview_drugs = pd.read_sql_query("SELECT id, name, types FROM drugs", conn)
conn.close()
print("=== clinpgx_drugs.db preview ===")
print(preview_drugs)